In [5]:
import torch
import torch.nn as nn

In [6]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.softmax = nn.Softmax(dim=-1)
        self.d_model = d_model
        self.num_heads = num_heads

    def forward(self, X):
        Q = self.q_proj(X)
        V = self.v_proj(X)
        K = self.k_proj(X)

        dim_model_single_head = self.d_model // self.num_heads

        list_Q = []
        list_V = []
        list_K = []

        for i in range(0, Q.shape[1], dim_model_single_head):
            list_Q.append(Q[:, i:i+dim_model_single_head])
            list_K.append(K[:, i:i+dim_model_single_head])
            list_V.append(V[:, i:i+dim_model_single_head])

        slices_with_attention = []

        for i in range(len(list_Q)):
            initial = torch.matmul(list_Q[i], list_K[i].T)
            
            normalized = self.softmax(torch.div(initial, torch.sqrt(torch.tensor(dim_model_single_head, dtype=torch.float32))))

            slices_with_attention.append(torch.matmul(normalized, list_V[i]))

        concatenated = torch.concat(slices_with_attention, dim=1)

        return self.out_proj(concatenated)

In [7]:
class FeedForward(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.proj_hidden_layer = nn.Linear(d_model, d_model * 4)
        self.proj_outer_layer = nn.Linear(d_model * 4, d_model)
        self.relu = nn.ReLU()

    def forward(self, X):
        hidden_layer = self.relu(self.proj_hidden_layer(X))

        return self.proj_outer_layer(hidden_layer)

In [8]:
class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))
        self.eps = eps

    def forward(self, X):
        matrix_mean = torch.mean(X, dim=-1, keepdims=True)
        matrix_standard_deviation = torch.std(X, dim=-1, keepdims=True)
        
        main_calculation = torch.div(torch.sub(X, matrix_mean), torch.add(matrix_standard_deviation, self.eps))

        scaled_calc = torch.mul(self.gamma, main_calculation)

        return torch.add(scaled_calc, self.beta)

In [9]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.multi_head_attention = MultiHeadAttention(d_model, num_heads)
        self.first_layer_norm = LayerNorm(d_model)
        self.second_layer_norm = LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model)

    def forward(self, X):
        multi_head = self.multi_head_attention(X)

        residual_1 = multi_head + X
        layer_norm_1 = self.first_layer_norm(residual_1)

        feed_forward = self.feed_forward(layer_norm_1)

        residual_2 = feed_forward + layer_norm_1
        layer_norm_2 = self.second_layer_norm(residual_2)

        return layer_norm_2

In [10]:
torch.manual_seed(42)
block = TransformerBlock(d_model=4, num_heads=2)

X = torch.tensor([[1, 0, 1, 0],
                [0, 1, 0, 1],
                [1, 1, 0, 0]], dtype=torch.float32)

output = block(X)
print(output)
print(output.shape)  # torch.Size([3, 4])

tensor([[ 1.0319, -0.9245,  0.6793, -0.7867],
        [-0.1009,  0.2623, -1.2865,  1.1251],
        [ 1.2396,  0.3666, -0.6635, -0.9427]], grad_fn=<AddBackward0>)
torch.Size([3, 4])
